# 05 — Historical GIS evidence audit and QGIS hand-off

**Role: Historical exposure screening and technical audit of observed GIS evidence.**

This notebook checks the validated `historical_exposure_screening` GeoPackage: **1 km mainland grid cells with fire recurrence measured in a 2 km context**. The evidence window is 2016–2025 observed ICNF burned-area history.

It does not create a prediction, probability, purchase decision, or recommendation category. The maps, descriptive tables, official comparison, and separate 2026 model estimate are presented once in [06_final_charts.ipynb](06_final_charts.ipynb).

In [1]:
from pathlib import Path
import json
import sys
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.notebook_support import resolve_project_root

PROJECT_ROOT = resolve_project_root(PROJECT_ROOT)
print('Repository root resolved; notebook paths are reported relative to it.')

Repository root resolved; notebook paths are reported relative to it.


## Optional controlled screening validation

The default cells inspect the published GeoPackage and its recorded validation metrics. Enable this switch only to recompute the bounded historical-screening attributes in memory and compare them with the existing output. It does not train a model, score 2026, or publish a replacement layer.

In [2]:
VALIDATE_HISTORICAL_SCREENING = False
if VALIDATE_HISTORICAL_SCREENING:
    from src.historical_exposure_screening import run_historical_exposure_screening
    screening_validation = run_historical_exposure_screening(validate_existing=True)
    print('Historical screening rerun matched the existing output:', screening_validation['deterministic_rerun']['analytical_values_exact'])
else:
    print('Using the validated published screening layer. Set VALIDATE_HISTORICAL_SCREENING=True for the bounded rerun check.')

Using the validated published screening layer. Set VALIDATE_HISTORICAL_SCREENING=True for the bounded rerun check.


## GeoPackage contract

This check confirms that the GIS layer is the expected reusable spatial evidence: correct layer, CRS, unique 1 km cell identifiers, fixed 2016–2025 observed-history window, and the documented 2 km outward recurrence context. It is an audit of stored evidence, not a visual interpretation or a model evaluation.

In [3]:
import pandas as pd
import pyogrio
from src.historical_exposure_screening import METRICS_PATH, OUTPUT_LAYER, OUTPUT_PATH

metrics = json.loads(METRICS_PATH.read_text(encoding='utf-8'))
layer_info = pyogrio.read_info(OUTPUT_PATH, layer=OUTPUT_LAYER)
attributes = pyogrio.read_dataframe(
    OUTPUT_PATH,
    layer=OUTPUT_LAYER,
    read_geometry=False,
    columns=[
        'cell_id', 'history_start_year', 'history_end_year',
        'fire_years_history_10y_2km', 'historical_exposure_band',
        'official_icnf_hazard_class',
    ],
)

assert metrics['no_predictive_claim']
assert layer_info['features'] == len(attributes) == 89_112
assert layer_info['crs'] == 'EPSG:3763'
assert attributes['cell_id'].is_unique
assert set(attributes['history_start_year']) == {2016}
assert set(attributes['history_end_year']) == {2025}

print({
    'path': OUTPUT_PATH.relative_to(PROJECT_ROOT).as_posix(),
    'layer': OUTPUT_LAYER,
    'crs': layer_info['crs'],
    'features': layer_info['features'],
    'history_window': '2016-2025',
    'recurrence_context': '2 km outward context around each 1 km cell',
})

{'path': 'data/processed/spatial_outputs/historical_residential_wildfire_exposure_screening.gpkg', 'layer': 'historical_exposure_screening', 'crs': 'EPSG:3763', 'features': 89112, 'history_window': '2016-2025', 'recurrence_context': '2 km outward context around each 1 km cell'}


## Summary and comparison audit

The underlying historical-band counts, official ICNF structural-hazard counts, and their cross-tab are recomputed from the GeoPackage and checked against the recorded validation metrics. The compact table below reports the audit outcome; the descriptive values and live visuals are intentionally shown only in Notebook 06 to avoid repetition.

In [4]:
band_order = ['lower', 'moderate', 'higher']
hazard_order = ['null', 'very_low', 'low', 'medium', 'high', 'very_high', 'unmatched']

band_summary = attributes.groupby('historical_exposure_band').size().reindex(band_order, fill_value=0)
hazard_summary = attributes.groupby('official_icnf_hazard_class').size().reindex(hazard_order, fill_value=0)
cross_tab = pd.crosstab(
    attributes['historical_exposure_band'], attributes['official_icnf_hazard_class']
).reindex(index=band_order, columns=hazard_order, fill_value=0)

expected_bands = pd.DataFrame(metrics['band_summary']).set_index('historical_exposure_band')
expected_hazards = pd.DataFrame(metrics['hazard_summary']).set_index('official_icnf_hazard_class')
pd.testing.assert_series_equal(band_summary, expected_bands['cell_count'], check_names=False)
pd.testing.assert_series_equal(hazard_summary, expected_hazards['cell_count'], check_names=False)
assert int(cross_tab.to_numpy().sum()) == 89_112

validation_status = pd.DataFrame([
    {'check': 'Historical exposure bands', 'result': 'Match the validated recorded counts', 'status': 'passed'},
    {'check': 'Official ICNF hazard classes', 'result': 'Match the validated recorded counts', 'status': 'passed'},
    {'check': 'Band × class cross-tab', 'result': 'Accounts for all 89,112 mainland cells', 'status': 'passed'},
    {
        'check': 'Historical recurrence thresholds',
        'result': f"Lower 0–{metrics['thresholds']['lower_max']}; moderate 2–{metrics['thresholds']['moderate_max']}; higher 4–10 years",
        'status': 'passed',
    },
])
display(validation_status)

,check,result,status
0,Historical exposure bands,Match the validated recorded counts,passed
1,Official ICNF hazard classes,Match the validated recorded counts,passed
2,Band × class cross-tab,"Accounts for all 89,112 mainland cells",passed
3,Historical recurrence thresholds,Lower 0–1; moderate 2–3; higher 4–10 years,passed


## QGIS and final-story hand-off

Open the historical GIS evidence in QGIS for interactive inspection. The QGIS project is a presentation of the same validated GeoPackage checked above; it is not a separate analysis.

- [Open the historical QGIS project](../qgis/wildfire_exposure_screening_portugal.qgz)
- [Read QGIS layer meanings and opening instructions](../qgis/README.md)
- [Read the screening and ICNF comparison validation report](../reports/validation/historical_exposure_screening_and_icnf_comparison.md)
- [Open the historical evidence GeoPackage](../data/processed/spatial_outputs/historical_residential_wildfire_exposure_screening.gpkg)
- [Continue to the final figures, descriptive tables, and separate 2026 estimate](06_final_charts.ipynb)

## Correct use

Lower historical exposure does not mean zero wildfire risk. This notebook validates observed 2016–2025 recurrence evidence for broad location comparison and shortlisting. It is not a next-year forecast, property-level safety guarantee, or purchase recommendation.